# 032 — GM record selection, AvgSA([0, 3]) — Stage 1 (compute)

The slow selection step. Runs the configurable multi-round selection + optimisation engine
(`build_final_ensembles`) to build the final per-`(site, iml)` record ensembles conditioned on
**AvgSA([0, 3])**, using the shared `SELECTION_CONFIG` (4 rounds dropping progressively more
causal-parameter bounds; round 4 is a shuffled-DB best-of retry).

**Incremental** — like nb `031`, this notebook calls `find_stale_stripes` and only (re)selects the
stripes that are missing or stale on disk. The stale check runs before any disagg data is read, and
only the shards of the sites that turn out to be stale are then loaded — an all-valid run opens none
of them. Per-stripe fingerprints name a single site's shard; the per-round stage caches and
`final_ensembles` cover a whole batch, so they fingerprint the digest of the entire shard set (the batch nb `031` just built gcim for). The round loop is
per-`(site, iml)` independent, so selecting a subset is equivalent to selecting those keys within the
full set. The batch is then **merged into `AvgSA_03_final_ensembles.pickle`** so that file stays the
COMPLETE `(site, iml)` set (it is read as complete by `summarise_record_availability.py` and
provenance-stamped by nb `040`). The canonical per-stripe record is written by nb `033`; the
per-round stage caches here are per-batch scratch. Set `FORCE_RECOMPUTE = True` to reselect every
wanted stripe.

**Upstream**:

| Input | Source |
|---|---|
| GCIM target distributions (complete set) | `gcim_dist_AvgSA_03.pickle` (nb `031`) |
| Combined selection GM database | `cfg["proc_data"]["gm_database"]` |
| IML-based disagg shards + poe stats | `cfg["proc_data"]["AvgSA_03_disagg_data_shards"]`, `cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| GMM logic tree + correlation flatfiles | loaded inside `setup_AvgSA03_gcim_gm_selection()` |

**Downstream** — `033-gm_selection_AvgSA_03_stage2_postprocess.ipynb` reads the complete
`AvgSA_03_final_ensembles.pickle`, writes the per-stripe pickles + manifests for the stale ones, and
re-verifies provenance.

**Run order** — run top to bottom (after `031`), then `033`. First run over a fresh results folder
selects everything (~an hour); subsequent "added a few IMLs" runs select only the new stripes.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    build_final_ensembles,
    stripe_input_fingerprint,
    find_stale_stripes,
)
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection,
    wanted_stripe_keys,
    SELECTION_CONFIG,
    stripe_source_fps,
)


cfg = load_config()

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:15: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
# canonical Stage-1 output (merged each run to stay the COMPLETE (site, iml) set)
final_ensembles_fp = cfg["proc_data"]["gm_selection"] / "AvgSA_03_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded; per-batch scratch).
# Named {im}_rd{ii}_{selection|optimisation}.pickle.
GMS = cfg["proc_data"]["gm_selection"]
IM = "AvgSA_03"
n_rounds = len(SELECTION_CONFIG["round_unbounded"])
stage_fps = {
    "select":   [GMS / f"{IM}_rd{ii}_selection.pickle"    for ii in range(1, n_rounds + 1)],
    "optimise": [GMS / f"{IM}_rd{ii}_optimisation.pickle" for ii in range(1, n_rounds + 1)],
}

# Inputs used for provenance fingerprinting (hashed by their file bytes). These stage /
# final artifacts cover a whole batch, so the disagg entry is the digest of the ENTIRE
# shard set (disagg_shard_dir -> disagg_shards_digest), not any one site's shard — see
# gm_selection._disagg_fingerprint_input.
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_shard_dir":  cfg["proc_data"]["AvgSA_03_disagg_data_shards"],
    "disagg_stats_file": cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"],
    "site_model_file":   cfg["hazard_models"]["eshm20_wp1_site_model"],
}

# The selection scheme (rounds, bounds, shuffle, seeds, rng_seed) now lives in the
# shared SELECTION_CONFIG (setup_AvgSA03_gm_selection.py) so 031/032/033 fingerprint
# each stripe identically. See that constant for the round-by-round description.

# other stuff:
rng_seed = SELECTION_CONFIG["rng_seed"]

# Escape hatch: force-reselect ALL wanted stripes, ignoring the per-stripe manifests
# (and bypassing every stage's staleness guard). Leave False for normal incremental
# runs: only the stale/new (site, iml) are (re)selected.
FORCE_RECOMPUTE = False

In [4]:
# Set up the selection context WITHOUT loading any disagg data (sites=()) — the stale
# check needs only the context and the index-derived key set. The shards for the stale
# sites are loaded in the next compute cell, once we know which sites those are.
_, disagg_stats, site_model, basic_selection_ctx, gm_db = setup_AvgSA03_gcim_gm_selection(sites=())

# Incremental cache: (re)select only stripes that are missing or stale on disk. Uses
# the same per-stripe fingerprint as 031/033 (excludes the IML subset JSON), so this
# matches the batch nb 031 just built gcim for.
RESULT_FOLDER = cfg["results"]["AvgSA_03_record_selection"]
source_fps_stripe = stripe_source_fps()
wanted = wanted_stripe_keys()
fp_fn = lambda s, i: stripe_input_fingerprint(
    s, i, source_fps_stripe, basic_selection_ctx, SELECTION_CONFIG)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)select.")

# load the (complete) gcim distributions written by nb 031
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print(f"Loaded gcim for {len(gcim_dists)} (site, iml).")
else:
    gcim_dists = {}
    print("No GCIM distribution data found — run nb 031 first.")

Built 0 (site, iml) disaggregations across 0 sites (0 of 60 shards loaded; 510 (site, iml) wanted in total).
510 wanted (site, iml): 510 already valid, 0 to (re)select.
Loaded gcim for 511 (site, iml).


## Stage 1 - Build final ensembles (slow compute)

Runs the configurable multi-round selection + optimisation engine and saves the canonical
`AvgSA_03_final_ensembles.pickle` (+ a `.manifest.json` provenance sidecar). Each round
drops progressively more causal-parameter bounds (`round_unbounded`), and each step is
provenance-cached: if an input (gm_db, gcim distributions, disagg, site model, params,
rng_seed, pickagm version, round config) changed since the cache was written, the stage is
recomputed instead of silently reusing stale data - set `FORCE_RECOMPUTE = True` to rebuild.

Rounds 1–3 (`force_optimisation = [False, True, False]`) reproduce the legacy AvgSA_03 result
exactly. **Round 4** re-optimises any `(site, poe)` still failing after round 3 over
`n_shuffles = 5` shuffled database orderings and keeps the best-scoring ensemble per
`(site, poe)` (replaced only if it passes or scores strictly better). Each round prints a
`[bounded: … | unbounded: …]` bounds label plus a selection-phase and optimisation-phase
breakdown; cached stages print `[cache] ... loaded`.

Post-processing (result pickles, plots, download/convert CSVs) lives in the separate,
fast notebook **`033-gm_selection_AvgSA_03_stage2_postprocess.ipynb`**.

In [5]:
# Force override: reselect every wanted stripe (ignores the per-stripe manifests).
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

# Capture the prior COMPLETE final_ensembles before build_final_ensembles overwrites
# the file with just this batch. On a forced full rebuild we start clean.
prior_final = {}
if final_ensembles_fp.is_file() and not FORCE_RECOMPUTE:
    with open(final_ensembles_fp, "rb") as f:
        prior_final = pickle.load(f)

# Subset to the stale batch and run the multi-round engine over just those keys. The
# round loop is per-(site, iml) independent, so selecting a subset is equivalent to
# selecting them within the full set. gcim for every batch key must be present (built
# by nb 031); a missing key means 031 was not run for this batch.
#
# The disagg shards are loaded HERE, and only for the sites with stale stripes — an
# all-valid run reads no disagg data at all.
if to_compute:
    stale_sites = {s for s, _ in to_compute}
    site_iml_disaggs, *_ = setup_AvgSA03_gcim_gm_selection(sites=stale_sites)
    batch = {k: site_iml_disaggs[k] for k in to_compute}
else:
    batch = {}

missing = [k for k in batch if k not in gcim_dists]
if missing:
    raise KeyError(
        f"{len(missing)} stale (site, iml) have no gcim — run nb 031 first. "
        f"e.g. {missing[:5]}")

if batch:
    batch_final = build_final_ensembles(
        batch,
        disagg_stats,
        gcim_dists,
        gm_db,
        basic_selection_ctx,
        site_model,
        source_fps=source_fps,
        stage_fps=stage_fps,
        output_fp=final_ensembles_fp,
        round_unbounded=SELECTION_CONFIG["round_unbounded"],
        force_optimisation=SELECTION_CONFIG["force_optimisation"],
        shuffle=SELECTION_CONFIG["shuffle"],
        n_shuffles=SELECTION_CONFIG["n_shuffles"],
        shuffle_rng_seeds=SELECTION_CONFIG["shuffle_rng_seeds"],
        rng_seed=rng_seed,
        force_recompute=FORCE_RECOMPUTE,
    )
else:
    batch_final = {}
    print("All stripes valid — nothing to (re)select.")

# MERGE the batch into the prior complete set so final_ensembles.pickle stays the
# COMPLETE (site, iml) selection: it is consumed as complete by
# summarise_record_availability.py and provenance-stamped by nb 040. Prune any key no
# longer wanted (an IML removed from a union list). The manifest build_final_ensembles
# wrote is source-file based, so it stays valid for the merged file.
wanted_set = set(wanted)
final_ensembles = {k: v for k, v in {**prior_final, **batch_final}.items() if k in wanted_set}
if batch_final or prior_final:
    with open(final_ensembles_fp, "wb") as f:
        pickle.dump(final_ensembles, f)
print(f"final_ensembles.pickle now holds {len(final_ensembles)} (site, iml) "
      f"({len(batch_final)} (re)selected this run).")

All stripes valid — nothing to (re)select.
final_ensembles.pickle now holds 510 (site, iml) (0 (re)selected this run).


In [6]:
# Isolate the (site, iml) that still fail the KS test after all rounds (across the full
# merged set), mapping each to its list of failing IMs. Not pickled - just a quick look.
failing_ensembles = {
    k: v["ks_failed_ims"]
    for k, v in final_ensembles.items()
    if v is not None and not v["ks_passed"]
}

print(f"{len(failing_ensembles)} (site, iml) still failing:")
for k in failing_ensembles:
    print("  ", k, "->", failing_ensembles[k])

0 (site, iml) still failing:


## One-off: reselect a stripe around an undownloadable record

**Deviation, deliberately localised — normally a no-op (`RESELECT_EXCLUDED = False`).**

A record that the selection picked turned out to be impossible to obtain from its source
database (the download crashes on it). It was selected in exactly **one** of the wanted
stripes. The fix is to reselect *that one stripe* against an in-memory-filtered copy of the
GM database, leaving the database file and every other stripe untouched.

The exclusion can name a single record or a whole **network** (as its list of RSNs, since the
combined DB carries no `Network` column). The 2026-08-27 case is network-level: NGA-Sub RSN
4040498 failed, and so did every other record of its network, **ONA** -- a single downhole
array whose flatfile entries are visibly unfinished and for which PEER serves no waveform at
all (0 of 28 ever downloaded, against ~40k working K-NET / KiK-net records). Excluding the
whole network is what stops the reselect from simply picking another broken sibling.

**Why not just delete the record from `ESM-NGAsub_combined.csv`** — three independent reasons:

1. `stripe_input_fingerprint` hashes `gm_db_file` **by its bytes**, so one edited byte makes
   *every* stripe stale: nbs `032`/`033` would reselect and rewrite all of them.
2. The selection results identify records by the database's positional `RangeIndex`
   (`greedy_optimise_ensemble` → `gm_db.loc[current_ensemble_indices]`). Deleting a CSV row
   shifts every row below it, silently invalidating the stripe pickles already on disk.
3. Round 4 shuffles with `filtered_gm_db.sample(frac=1, random_state=rng_seed)`, and a pandas
   permutation depends on the frame **length** — removing one row reshuffles the whole
   ordering, so already-analysed stripes could come back with different record sets.

`drop_unavailable_records` (see `setup_AvgSA03_gm_selection.UNAVAILABLE_RECORDS`) therefore
filters a **copy in memory**, using `.drop(index=...)` so the surviving rows keep their
original index labels. `source_fps` stays **canonical**: that keeps this stripe's manifest
consistent with all the others and stops it reading as permanently stale on every future run.
The trade-off is the one recorded inaccuracy — this stripe's manifest claims it came from the
full database. The `db_exclusions` key written into the ensemble below is the machine-readable
record of the truth.

**Run order (important)**

1. Run this notebook top to bottom with `RESELECT_EXCLUDED = False`; the stale check must
   report **0 to (re)select**.
2. Set `RESELECT_EXCLUDED = True` and run *only* the cell below.
3. *Then* delete the target stripe's `...__gm_selection.pickle` **and** its `.manifest.json`
   from `cfg["results"]["AvgSA_03_record_selection"]` — deleting them any earlier would make
   the normal compute cell above reselect the stripe from the **full** database.
4. Run nb `033` (rewrites the one missing stripe + the download/convert CSVs), then nb `040`.

Note this cell overwrites the `AvgSA_03_rd{1..4}_*.pickle` stage caches — they are documented
per-batch scratch, so that is harmless. Leave `FORCE_RECOMPUTE = False` throughout.


In [8]:
# Reselect ONE stripe against an in-memory-filtered copy of the GM database. Off by
# default, so a top-to-bottom rerun of this notebook is a no-op. See the markdown above
# for why the database file itself must not be edited, and for the run order.
RESELECT_EXCLUDED = True

# The (site, iml) stripe that contains the undownloadable record. The record(s)
# themselves are declared in setup_AvgSA03_gm_selection.UNAVAILABLE_RECORDS.
# 2026-08-27: NGA-Sub RSN 4040498 was selected in exactly one of the 510 stripes.
TARGET = (31, 1.15)

if RESELECT_EXCLUDED:
    from phd_project.scripts.WP1_ground_motion_set.gm_selection import stripe_pickle_path
    from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
        UNAVAILABLE_RECORDS,
        drop_unavailable_records,
    )

    # --- guards ----------------------------------------------------------------
    if TARGET is None:
        raise ValueError("Set TARGET to the (site, iml) stripe to reselect.")
    if TARGET not in set(wanted):
        raise KeyError(f"{TARGET} is not a wanted (site, iml); check the iml value.")
    if not UNAVAILABLE_RECORDS:
        raise ValueError(
            "UNAVAILABLE_RECORDS is empty - declare the undownloadable record in "
            "setup_AvgSA03_gm_selection.py first.")
    if to_compute:
        # A stale stripe means the normal compute cell above still has work to do; it
        # would reselect from the FULL database. Resolve that before deviating.
        raise RuntimeError(
            f"{len(to_compute)} stripe(s) are stale - run the normal compute cell "
            f"first, then come back here. e.g. {to_compute[:5]}")

    # --- filtered database (the file on disk is NEVER touched) ------------------
    gm_db_excl = drop_unavailable_records(gm_db)
    dropped_idx = gm_db.index.difference(gm_db_excl.index)
    if len(dropped_idx) == 0:
        raise RuntimeError(
            "No rows were dropped - the identity fields in UNAVAILABLE_RECORDS do not "
            "match any row of the combined database.")

    # --- reselect just this stripe ---------------------------------------------
    site, iml = TARGET
    site_iml_disaggs, *_ = setup_AvgSA03_gcim_gm_selection(sites={int(site)})
    batch = {TARGET: site_iml_disaggs[TARGET]}
    if TARGET not in gcim_dists:
        raise KeyError(f"No gcim for {TARGET} - run nb 031 first.")

    # Capture the COMPLETE set before build_final_ensembles overwrites the file with
    # just this one-key batch.
    with open(final_ensembles_fp, "rb") as f:
        prior_final = pickle.load(f)

    # Refuse to run against an already-truncated artifact. build_final_ensembles
    # replaces this file with its batch, so a previous failed one-off run can leave a
    # 1-key file behind; merging onto that would silently discard the other 509 (and so
    # would the restore in the except clause below). Recover first --
    # `dvc checkout --force data_processed/06_gm_selection/AvgSA_03_final_ensembles.pickle.dvc`
    # -- then re-run.
    if len(prior_final) != len(wanted):
        raise RuntimeError(
            f"final_ensembles.pickle holds {len(prior_final)} (site, iml), expected "
            f"{len(wanted)}. It is truncated - restore it before reselecting.")

    # The in-memory filter is INVISIBLE to the provenance fingerprint: source_fps hashes
    # the database FILE, which is unchanged, and every other input to this stripe is
    # identical too. A stage cached by a run that used the full (or a differently
    # filtered) database therefore looks perfectly valid and gets handed straight back --
    # complete with the record we are trying to exclude. Two defences:
    #   * dedicated stage paths, so the main per-batch scratch caches are neither read
    #     nor overwritten by this deviation;
    #   * force_recompute=True, so no staleness guard can short-circuit any round.
    excl_stage_fps = {
        "select":   [GMS / f"{IM}_excl_rd{ii}_selection.pickle"
                     for ii in range(1, n_rounds + 1)],
        "optimise": [GMS / f"{IM}_excl_rd{ii}_optimisation.pickle"
                     for ii in range(1, n_rounds + 1)],
    }

    # source_fps stays CANONICAL on purpose (see the markdown above): it keeps this
    # stripe's manifest consistent with the other 509 and stops it reading as
    # permanently stale on every future run.
    #
    # NOTE: build_final_ensembles OVERWRITES final_ensembles_fp with just this one-key
    # batch before it returns, so from here until the merge below the canonical file on
    # disk is INCOMPLETE. Everything after the call therefore runs under a try/except
    # that puts the complete prior_final back if anything raises.
    batch_final = build_final_ensembles(
        batch,
        disagg_stats,
        gcim_dists,
        gm_db_excl,
        basic_selection_ctx,
        site_model,
        source_fps=source_fps,
        stage_fps=excl_stage_fps,
        output_fp=final_ensembles_fp,
        round_unbounded=SELECTION_CONFIG["round_unbounded"],
        force_optimisation=SELECTION_CONFIG["force_optimisation"],
        shuffle=SELECTION_CONFIG["shuffle"],
        n_shuffles=SELECTION_CONFIG["n_shuffles"],
        shuffle_rng_seeds=SELECTION_CONFIG["shuffle_rng_seeds"],
        rng_seed=rng_seed,
        force_recompute=True,
    )

    try:
        ens = batch_final[TARGET]
        if ens is None:
            raise RuntimeError(f"No ensemble could be formed for {TARGET} without the "
                               f"excluded record(s).")

        # --- record the deviation inside the stripe ------------------------------
        # Invisible to the fingerprint, and it survives into the stripe pickle because
        # nb 033 mutates and re-pickles this same dict.
        ens["db_exclusions"] = [dict(r) for r in UNAVAILABLE_RECORDS]

        # --- checks ---------------------------------------------------------------
        still_there = ens["recs"].index.intersection(dropped_idx)
        if len(still_there):
            raise RuntimeError(f"Excluded record still selected: {list(still_there)}")

        prev = prior_final.get(TARGET)
        prev_recs = set(prev["recs"].index) if prev is not None else set()
        new_recs = set(ens["recs"].index)
        print(f"\n{TARGET}: {len(new_recs)} records, ks_passed = {ens['ks_passed']}, "
              f"{len(new_recs - prev_recs)} of them new vs the previous ensemble.")

        # --- merge back into the COMPLETE set -------------------------------------
        wanted_set = set(wanted)
        final_ensembles = {k: v for k, v in {**prior_final, **batch_final}.items()
                           if k in wanted_set}
        with open(final_ensembles_fp, "wb") as f:
            pickle.dump(final_ensembles, f)
        print(f"final_ensembles.pickle now holds {len(final_ensembles)} (site, iml) "
              f"(was {len(prior_final)}).")

        # --- next step: make nb 033 rewrite this stripe ---------------------------
        # Its manifest is still canonically valid, so 033 would otherwise skip it.
        # Delete both files by hand, THEN run 033.
        stripe_fp = stripe_pickle_path(RESULT_FOLDER, site, iml)
        print("\nNow delete these two files, then run nb 033:")
        print(f"  {stripe_fp}")
        print(f"  {stripe_fp.with_name(stripe_fp.name + '.manifest.json')}")

    except Exception:
        # build_final_ensembles left the one-key batch on disk; put the complete set
        # back so a failure here can never truncate the canonical artifact.
        with open(final_ensembles_fp, "wb") as f:
            pickle.dump(prior_final, f)
        print(f"\n!! failed - restored the complete {len(prior_final)}-key "
              f"final_ensembles.pickle")
        raise


  excluding NGASub NGA-Sub network ONA (28 RSNs, 4040476-4040503): 48 row(s) [68473, 68474, 68475, 68476, 68477, 68478, 68479, 68480, 68481, 68482, 68483, 68484, 68485, 68486, 68487, 68488, 68489, 68490, 68491, 68492, 68493, 68494, 68495, 68496, 112550, 112551, 112552, 112553, 112554, 112555, 112556, 112557, 112558, 112559, 112560, 112561, 112562, 112563, 112564, 112565, 112566, 112567, 112568, 112569, 112570, 112571, 112572, 112573]
gm_db: 128816 -> 128768 rows (48 dropped)
Built 7 (site, iml) disaggregations across 1 sites (1 of 60 shards loaded; 510 (site, iml) wanted in total).

================ Round 1/4 ================
Bounds  [bounded: m, d, vs30 | unbounded: none]
Selection phase — all 1 (site, iml) [first round]


R1/4 select (1 sites):   0%|          | 0/1 [00:00<?, ?it/s]

    -> 1 pass, 0 fail KS
Optimisation phase — 0 (site, iml)  [only sites still failing]
-> all (site, iml) now pass

================ Round 2/4 ================
Bounds  [bounded: m, vs30 | unbounded: d]
Selection phase — none need reselection (every failing (site, iml) already has a record set)
Optimisation phase — 0 (site, iml)  [only sites still failing]
-> all (site, iml) now pass

================ Round 3/4 ================
Bounds  [bounded: none | unbounded: m, d, vs30]
Selection phase — none need reselection (every failing (site, iml) already has a record set)
Optimisation phase — 0 (site, iml)  [only sites still failing]
-> all (site, iml) now pass

================ Round 4/4  (shuffled DB, best of 5) ================
Bounds  [bounded: none | unbounded: m, d, vs30]
Selection phase — none need reselection (every failing (site, iml) already has a record set)
Optimisation phase — 0 (site, iml)  [only sites still failing]
-> all (site, iml) now pass

OK! - all (site, iml) have a pas